# Cosmic web v0 — mapping voids with the tidal-tensor (T-web) method

A minimal, runnable first step toward a 3D map of the cosmic web. We segment a density field into four classes — **void, sheet (wall), filament, knot (cluster)** — using the classical tidal-tensor classifier. No quantum, no GPU required; the whole thing runs in a few seconds in Colab.

**The idea.** Take a density field δ(x). Solve for the potential (∇²φ = δ), take its Hessian (the tidal/deformation tensor), and at each point find the three eigenvalues. Count how many exceed a threshold λ_th:

| eigenvalues > λ_th | collapsing axes | class |
|---|---|---|
| 0 | none → expanding everywhere | **void** |
| 1 | one | **sheet / wall** |
| 2 | two | **filament** |
| 3 | all three | **knot / cluster** |

This is exactly the labeling that modern ML void-finders (e.g. DeepVoid) feed to a 3D U-Net. We start on a *synthetic* field so it runs immediately, then there's a clearly marked section to swap in real QUIJOTE simulation data.

*Refs:* QUIJOTE simulations (`quijote-simulations.readthedocs.io`), Pylians (`github.com/franciscovillaescusa/Pylians3`), DeepVoid (arXiv:2504.21134).

## 0. Parameters — your one place to tweak

Small project, single knob-box. Lower `N` for faster iteration; raise it for finer maps. `lam_th` controls the void/filament balance (higher → more void volume).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

# --- config ---
N        = 128      # grid cells per side (128 -> ~6s; try 64 while experimenting)
L        = 250.0    # box size in Mpc/h
sigma_g  = 1.2      # rms of the underlying Gaussian field (sets how 'clumpy')
R_smooth = 3.0      # smoothing scale in Mpc/h (the scale at which the web is defined)
lam_th   = 0.3      # eigenvalue threshold (0.2-0.4 typical; higher = more void)
seed     = 42

CLASS_NAMES  = ['void', 'sheet', 'filament', 'knot']
CLASS_COLORS = ['#08306b', '#6baed6', '#fd8d3c', '#cb181d']  # cold/empty -> hot/dense

## 1. A synthetic cosmic-web density field

We build a Gaussian random field with a CDM-like power spectrum (BBKS transfer function), then apply a lognormal transform so the density is positive with deep voids and dense peaks — a cheap but convincing stand-in for the real thing.

In [ ]:
def Pk_bbks(k, ns=0.96, Gamma=0.2):
    '''Crude CDM power spectrum: P(k) = k^ns * T(k)^2 (BBKS transfer function).'''
    k = np.asarray(k, dtype=float)
    q = np.where(k > 0, k / Gamma, 1e-10)
    T = (np.log(1 + 2.34*q) / (2.34*q)) * \
        (1 + 3.89*q + (16.1*q)**2 + (5.46*q)**3 + (6.71*q)**4)**(-0.25)
    return k**ns * T**2

def make_lognormal_field(N, L, sigma_g=1.0, seed=42):
    '''Return a lognormal density-contrast field delta(x) on an N^3 grid.'''
    kax = np.fft.fftfreq(N, d=L/N) * 2*np.pi          # angular wavenumbers (h/Mpc)
    KX, KY, KZ = np.meshgrid(kax, kax, kax, indexing='ij')
    kmag = np.sqrt(KX**2 + KY**2 + KZ**2); kmag[0,0,0] = 1.0
    rng = np.random.default_rng(seed)
    white_k = np.fft.fftn(rng.standard_normal((N, N, N)))
    Pk = Pk_bbks(kmag); Pk[0,0,0] = 0.0
    g = np.fft.ifftn(white_k * np.sqrt(Pk)).real      # Gaussian field with P(k)
    g -= g.mean(); g *= sigma_g / g.std()
    sg = g.std()
    return np.exp(g - 0.5*sg**2) - 1.0                # lognormal -> mean(1+delta)=1

delta = make_lognormal_field(N, L, sigma_g=sigma_g, seed=seed)
print(f"delta: min={delta.min():.2f}  max={delta.max():.1f}  mean={delta.mean():.3f}  std={delta.std():.2f}")

## 2. The T-web classifier

Three small functions: smooth the field, build the tidal tensor by FFT, then count eigenvalues above threshold. The tensor is normalised so its trace equals δ (the eigenvalues sum to the density contrast) — a handy sanity check.

In [ ]:
def smooth_gauss(delta, L, R):
    N = delta.shape[0]
    kax = np.fft.fftfreq(N, d=L/N) * 2*np.pi
    KX, KY, KZ = np.meshgrid(kax, kax, kax, indexing='ij')
    k2 = KX**2 + KY**2 + KZ**2
    return np.fft.ifftn(np.fft.fftn(delta) * np.exp(-0.5 * k2 * R**2)).real

def tidal_tensor(delta, L):
    '''Deformation tensor T_ij = d^2 phi / dx_i dx_j, with trace(T) = delta.'''
    N = delta.shape[0]
    kax = np.fft.fftfreq(N, d=L/N) * 2*np.pi
    KX, KY, KZ = np.meshgrid(kax, kax, kax, indexing='ij')
    Kv = [KX, KY, KZ]
    k2 = KX**2 + KY**2 + KZ**2; k2[0,0,0] = 1.0
    dk = np.fft.fftn(delta)
    T = np.zeros((N, N, N, 3, 3))
    for i in range(3):
        for j in range(i, 3):
            Tij = np.fft.ifftn((Kv[i]*Kv[j]/k2) * dk).real
            T[..., i, j] = Tij
            if i != j: T[..., j, i] = Tij
    return T

def classify_web(T, lam_th=0.0):
    eigs = np.linalg.eigvalsh(T)                 # ascending eigenvalues, shape (N,N,N,3)
    n_above = np.sum(eigs > lam_th, axis=-1)     # 0..3  ==  class index
    return n_above.astype(np.int8), eigs

delta_s = smooth_gauss(delta, L, R_smooth)
T       = tidal_tensor(delta_s, L)
labels, eigs = classify_web(T, lam_th=lam_th)

print(f"trace check (max |sum(eig) - delta_s|): {np.abs(eigs.sum(-1) - delta_s).max():.1e}\n")
for c in range(4):
    print(f"  {CLASS_NAMES[c]:9s}: {100*(labels==c).mean():5.1f}% of volume")

## 3. Look at the map

A central 2D slice: the density on the left, the four-class web on the right. Voids should fill most of the area, with filaments threading between rare knots.

In [ ]:
z = N // 2
dens_slice  = np.log10(np.clip(1 + delta, 1e-3, None))[:, :, z]
label_slice = labels[:, :, z]

cmap = ListedColormap(CLASS_COLORS)
norm = BoundaryNorm([-.5, .5, 1.5, 2.5, 3.5], cmap.N)

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
im0 = ax[0].imshow(dens_slice.T, origin='lower', cmap='inferno',
                   extent=[0, L, 0, L], vmin=np.percentile(dens_slice, 2))
ax[0].set_title(r'density   log$_{10}$(1+$\delta$)')
plt.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)

im1 = ax[1].imshow(label_slice.T, origin='lower', cmap=cmap, norm=norm, extent=[0, L, 0, L])
ax[1].set_title('cosmic-web class')
cb = plt.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.04, ticks=[0, 1, 2, 3])
cb.ax.set_yticklabels(CLASS_NAMES)

for a in ax:
    a.set_xlabel('Mpc/h'); a.set_ylabel('Mpc/h')
plt.tight_layout(); plt.show()

## 4. Swap in real QUIJOTE data

Same pipeline, real particles. Steps:

1. Make a free **Globus** account and grab **one** fiducial QUIJOTE snapshot (snapshot `004` ≈ z=0). See `quijote-simulations.readthedocs.io`.
2. Install Pylians (next cell).
3. Point `SNAP` at the downloaded file and run the cell after.

Everything downstream — `smooth_gauss`, `tidal_tensor`, `classify_web`, and the plot — is reused unchanged.

In [ ]:
# Run once. Compiles C/Cython, ~1-3 min. If it fails, see:
# https://github.com/franciscovillaescusa/Pylians3
!pip install Pylians

In [ ]:
import os
# QUIJOTE snapshot numbering: 000->z=3, 001->z=2, 002->z=1, 003->z=0.5, 004->z=0
SNAP = "/content/quijote/fiducial/0/snapdir_004/snap_004"   # <-- edit me

if not os.path.exists(SNAP):
    print("No QUIJOTE snapshot at:", SNAP)
    print("Download one via Globus, set SNAP above, then re-run this cell.")
else:
    import readgadget, MAS_library as MASL
    head    = readgadget.header(SNAP)
    BoxSize = head.boxsize / 1e3                       # kpc/h -> Mpc/h
    pos     = readgadget.read_block(SNAP, "POS ", [1]) / 1e3   # CDM positions, Mpc/h
    print(f"loaded {pos.shape[0]:,} particles, BoxSize = {BoxSize:.0f} Mpc/h")

    delta_real = np.zeros((N, N, N), dtype=np.float32)
    MASL.MA(pos.astype(np.float32), delta_real, BoxSize, 'CIC')   # cloud-in-cell
    delta_real = delta_real / delta_real.mean() - 1.0

    delta_s_r = smooth_gauss(delta_real.astype(float), BoxSize, R_smooth)
    T_r       = tidal_tensor(delta_s_r, BoxSize)
    labels_r, _ = classify_web(T_r, lam_th=lam_th)
    for c in range(4):
        print(f"  {CLASS_NAMES[c]:9s}: {100*(labels_r==c).mean():5.1f}%")
    # to visualise: set delta=delta_real, labels=labels_r, L=BoxSize and rerun the Step-3 cell

## Next steps

- **Tune & explore.** Slide `lam_th`, `R_smooth`, `sigma_g`; watch the volume fractions and morphology shift. Real N-body data gives crisper, more anisotropic filaments than the lognormal mock.
- **The ML step (the actual research question).** Train a 3D U-Net to predict this label cube from a *sparser* input — a halo-count field or subsampled density — and score it with F1 / IoU / Matthews correlation coefficient against the ground truth. That's the DeepVoid recipe on your own data.
- **Cross-check the voids.** Run Pylians' spherical void finder (`void_library`) or VIDE (`bitbucket.org/cosmicvoids/vide_public`) on the same field and compare to the void class here.
- **Go observational.** Apply the trained model to a DESI DR1 region (`data.desi.lbl.gov/public/dr1`), or run VIDE directly on the DESI large-scale-structure catalog for a real void map.